In [ ]:
# Compare exact Parquet quantiles with multiple approximation runs.
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Inputs: update paths to point at your data.
# PARQUET_UID_DIR = "pyspark_out/itap/itap-uid-sorted-parquet"
# PARQUET_GID_DIR = "pyspark_out/itap/itap-gid-sorted-parquet"
PARQUET_UID_DIR = "pyspark_out/nersc/nersc-uid-sorted-parquet"
PARQUET_GID_DIR = "pyspark_out/nersc/nersc-gid-sorted-parquet"
QUANTILES = { # 1/2
    "p10": 0.10,
    "p25": 0.25,
    "p50": 0.50,
    "p75": 0.75,
    "p90": 0.90,
    "p99": 0.99,
}

In [ ]:
def load_uid_gid_parquet(uid_dir: str, gid_dir: str) -> pd.DataFrame:
    """Load uid and gid parquet datasets, stack into one DataFrame with typ/id/metric/value."""

    def _load(parquet_dir: str, typ: str, id_col: str) -> pd.DataFrame:
        dataset = pq.ParquetDataset(parquet_dir)
        table = dataset.read()
        df = table.to_pandas()
        df = df.copy()
        df["typ"] = typ
        df["id"] = df[id_col].astype("int64")
        df["metric"] = df["metric"].astype(str)
        df["value"] = pd.to_numeric(df["value"], errors="coerce").astype("int64")
        return df[["typ", "id", "metric", "value"]]

    uid_df = _load(uid_dir, "uid", "uid")
    gid_df = _load(gid_dir, "gid", "gid")
    combined = pd.concat([uid_df, gid_df], ignore_index=True)
    return combined

In [ ]:
# Load combined uid+gid parquet into a single normalized DataFrame.
df = load_uid_gid_parquet(PARQUET_UID_DIR, PARQUET_GID_DIR)
print(f"Combined df: {df.shape}")


In [ ]:
df

In [ ]:
# def sketches_csvs(abbv): # dd, kll, req, tdigest
#     return [
#         f"itap-approx/itap-uid-{abbv}-1.csv",
#         f"itap-approx/itap-uid-{abbv}-2.csv",
#         f"itap-approx/itap-uid-{abbv}-3.csv",
#         f"itap-approx/itap-gid-{abbv}-1.csv",
#         f"itap-approx/itap-gid-{abbv}-2.csv",
#         f"itap-approx/itap-gid-{abbv}-3.csv",
#     ]
def sketches_csvs(abbv): # dd, kll, req, td
    return [
        f"nersc-approx/uid-{abbv}-1.csv",
        f"nersc-approx/uid-{abbv}-2.csv",
        f"nersc-approx/uid-{abbv}-3.csv",
        f"nersc-approx/gid-{abbv}-1.csv",
        f"nersc-approx/gid-{abbv}-2.csv",
        f"nersc-approx/gid-{abbv}-3.csv",
    ]
sketches_csv = sketches_csvs("dd")


In [ ]:
# Load all approximation CSVs (uid + gid runs) into one DataFrame.
def load_all_ddsketch(paths):
    rows = []
    for path in paths:
        df = pd.read_csv(path)
        run = Path(path).stem
        typ = "uid" if "uid" in run else "gid"
        id_col = "uid" if typ == "uid" else "gid"
        df = df.rename(columns={id_col: "id"})
        df["id"] = df["id"].astype("int64")
        df["metric"] = df["metric"].astype(str)
        df["typ"] = typ
        df["run"] = run
        rows.append(
            df[["typ", "id", "metric", "run", "p10", "p25", "p50", "p75", "p90", "p99"]] # 2/2
        )
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


approx_df = load_all_ddsketch(sketches_csv)
print(f"Approx df: {approx_df.shape}")

In [ ]:
approx_df

In [ ]:
# Compute rank/value errors per (typ, id, metric, run) using exact + approx inputs.
def compute_rank_errors_by_typ_run(
    exact_df: pd.DataFrame, approx_df: pd.DataFrame, quantiles=QUANTILES
) -> pd.DataFrame:
    """Return per-quantile error rows with exact/approx ranks and value deltas."""
    exact_df = exact_df.copy()
    exact_df["typ"] = exact_df["typ"].astype(str)
    exact_df["id"] = exact_df["id"].astype("int64")
    exact_df["metric"] = exact_df["metric"].astype(str)
    exact_df["value"] = pd.to_numeric(exact_df["value"], errors="coerce").astype(
        "int64"
    )

    approx_df = approx_df.copy()
    approx_df["typ"] = approx_df["typ"].astype(str)
    approx_df["id"] = approx_df["id"].astype("int64")
    approx_df["metric"] = approx_df["metric"].astype(str)
    approx_df["run"] = approx_df["run"].astype(str)
    approx_indexed = approx_df.set_index(["typ", "id", "metric", "run"])

    results = []
    for (typ, id_val, metric), values in exact_df.groupby(["typ", "id", "metric"])[
        "value"
    ]:
        try:
            approx_rows = approx_indexed.loc[(typ, id_val, metric)]
        except KeyError:
            continue  # no approx rows for this key

        if isinstance(approx_rows, pd.Series):
            approx_rows = approx_rows.to_frame().T

        vals = np.sort(values.to_numpy())
        N = len(vals)
        for run, approx_row in approx_rows.iterrows():
            for col, p in quantiles.items():
                if col not in approx_row:
                    continue
                q_hat = int(approx_row[col])
                idx = int(p * (N - 1))
                q_exact = int(vals[idx])

                rank_hat = int(np.searchsorted(vals, q_hat, side="right"))
                expected_rank = idx + 1
                rank_error = abs(rank_hat - expected_rank)
                rank_error_norm = rank_error / N

                abs_val_err = abs(q_hat - q_exact)
                rel_value_error = abs_val_err / abs(q_exact) if q_exact != 0 else np.nan

                results.append(
                    {
                        "typ": typ,
                        "id": int(id_val),
                        "metric": metric,
                        "run": run,
                        "quantile": col,
                        "p": p,
                        "exact": q_exact,
                        "approx": q_hat,
                        "rank_hat": rank_hat,
                        "expected_rank": expected_rank,
                        "rank_error": rank_error,
                        "rank_error_norm": rank_error_norm,
                        "N": N,
                        "abs_val_err": abs_val_err,
                        "rel_value_error": rel_value_error,
                    }
                )
    return pd.DataFrame(results)


rank_errors = compute_rank_errors_by_typ_run(df, approx_df)

In [ ]:
rank_errors

In [ ]:
# Aggregate across runs; default groups by typ/id/metric/quantile.
def summarize_run_errors(rank_errors: pd.DataFrame, group_keys=None, min_n: int | None = None) -> pd.DataFrame:
    if rank_errors.empty:
        return rank_errors
    if min_n is not None:
        rank_errors = rank_errors[rank_errors["N"] >= min_n]
        if rank_errors.empty:
            return rank_errors
    keys = group_keys if group_keys is not None else ["typ", "id", "metric", "quantile"]
    grouped = (
        rank_errors.groupby(keys)
        .agg(
            run_count=("run", "nunique"),
            # approx_mean=("approx", "mean"),
            # approx_std=("approx", "std"),
            # rank_error_mean=("rank_error", "mean"),
            # rank_error_max=("rank_error", "max"),
            rank_error_norm_mean=("rank_error_norm", "mean"),
            rank_error_norm_max=("rank_error_norm", "max"),
            # abs_val_err_mean=("abs_val_err", "mean"),
            # abs_val_err_max=("abs_val_err", "max"),
            rel_value_error_mean=("rel_value_error", "mean"),
            rel_value_error_max=("rel_value_error", "max"),
            # exact_first=("exact", "first"),
            # N_first=("N", "first"),
        )
        .reset_index()
    )
    return grouped


grouped_rank_errors = summarize_run_errors(rank_errors, group_keys=["quantile"], min_n=100)
# For quantile-only view, use:
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
grouped_rank_errors
